# 06 - Frequency Modeling

**Purpose:** Select the predictors for the claim frequency model in each of the
four coverages, and fit the final Poisson GLM. Candidate variables are added by
forward stepwise selection, scored on out-of-sample Poisson deviance against the
holdout set. The distribution family is taken from 04; this notebook decides the
mean structure only.

**Input:** `data/processed/frequency_data.csv`
**Output:** Fitted frequency GLMs and their relativities, carried into
08_RatingAlgorithm.
**Depends on:** 02_Cleaned_Data, 04_Frequency_Distribution

**Specification:** Poisson GLM, log link, exposure = 1 policy-year per
student-coverage so no offset is required.

**Candidate predictors:** class_year, study, gpa, greek, gender, off_campus,
distance_to_campus, sprinklered. Excluded: student_id and name (identifiers with
no causal content), coverage (models are fitted separately by coverage),
risk_tier.

**Author:** Dalton | **Last run:** 2026-09-27

First we need to analyze what predictor variables are significant,(not analyzing name, id, and its already grouped by coverage). also not using risk_tier becuase thats applied after underwriting. Model trained only on data morked training(not holdout)

In [26]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
PROC = ROOT / "data" / "processed"

freq   = pd.read_csv(PROC / "frequency_data.csv")   # 40,000 student-coverage rows
claims = pd.read_csv(PROC / "claims.csv")           # 1,819 individual claims

pd.set_option('display.max_colwidth', None)

# 'class' is a Python keyword, so patsy formulas cannot use it
freq   = freq.rename(columns={'class': 'class_year'})
claims = claims.rename(columns={'class': 'class_year'})

COVERAGES = ['Personal Property', 'Additional Living Expense',
             'Guest Medical', 'Liability']

# training only; holdout is sealed until validation
fq_train = {c: freq[(freq.coverage == c) &
                    (freq.training_holdout == 'training')].reset_index(drop=True)
            for c in COVERAGES}

cl_train = {c: claims[(claims.coverage == c) &
                      (claims.training_holdout == 'training')].reset_index(drop=True)
            for c in COVERAGES}

RATING_VARS = ['class_year', 'study', 'gpa', 'greek', 'gender',
               'off_campus', 'distance_to_campus', 'sprinklered']

pd.DataFrame({
    'exposures': {c: len(fq_train[c]) for c in COVERAGES},
    'claims':    {c: len(cl_train[c]) for c in COVERAGES},
    'frequency': {c: fq_train[c].claim_count.mean().round(4) for c in COVERAGES},
})

,exposures,claims,frequency
Personal Property,7999,787,0.0984
Additional Living Expense,7999,427,0.0534
Guest Medical,7999,187,0.0234
Liability,7999,67,0.0084


seperated training and holdout. renamed class to classyear to avoid python conflicts with class

Personal property

In [2]:
pp = fq_train['Personal Property'][RATING_VARS + ['claim_count']].copy()

for c in ['class_year', 'study', 'greek', 'gender', 'off_campus']:
    pp[c] = pp[c].astype('category')

pp.head(10)

,class_year,study,gpa,greek,gender,off_campus,distance_to_campus,sprinklered,claim_count
0,Junior,Humanities,1.203913,Non-greek,Male,On campus,0.000000,True,0
1,Junior,Science,0.609616,Non-greek,Female,On campus,0.000000,True,0
2,Junior,Business,3.073385,Non-greek,Female,Off campus,0.223942,False,0
3,Senior,Business,3.224736,Non-greek,Male,On campus,0.000000,False,0
4,Sophomore,Business,2.687851,Non-greek,Female,On campus,0.000000,False,0
5,Junior,Business,0.688659,Non-greek,Other,On campus,0.000000,True,0
6,Junior,Science,3.243341,Non-greek,Female,On campus,0.000000,False,0
7,Junior,Humanities,3.317148,Non-greek,Female,Off campus,0.467159,True,0
8,Freshman,Business,1.309350,Greek,Other,On campus,0.000000,True,0
9,Sophomore,Other,0.981003,Non-greek,Female,Off campus,10.083255,True,1


got relevant data and added 

In [3]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# holdout frame for Personal Property, same treatment as the training frame
pp_hold = freq[(freq.coverage == 'Personal Property') &
               (freq.training_holdout == 'holdout')].copy()
for c in ['class_year', 'study', 'greek', 'gender', 'off_campus']:
    pp_hold[c] = pp_hold[c].astype('category')


def poisson_deviance(y, mu):
    """2 * sum[ y*log(y/mu) - (y - mu) ], with 0*log(0) = 0."""
    y   = np.asarray(y,  dtype=float)
    mu  = np.asarray(mu, dtype=float)
    pos = y > 0
    term = np.zeros_like(y)
    term[pos] = y[pos] * np.log(y[pos] / mu[pos])
    return 2.0 * np.sum(term - (y - mu))


rows = []

# baseline: intercept only
null = smf.glm('claim_count ~ 1', data=pp, family=sm.families.Poisson()).fit()
rows.append(('(null)', null.aic,
             poisson_deviance(pp_hold.claim_count, null.predict(pp_hold))))

# one model per candidate predictor
for v in RATING_VARS:
    m = smf.glm(f'claim_count ~ {v}', data=pp, family=sm.families.Poisson()).fit()
    rows.append((v, m.aic,
                 poisson_deviance(pp_hold.claim_count, m.predict(pp_hold))))

step1 = (pd.DataFrame(rows, columns=['predictor', 'train_AIC', 'holdout_deviance'])
           .sort_values('holdout_deviance', ascending=False)
           .reset_index(drop=True))

step1.round(3)

,predictor,train_AIC,holdout_deviance
0,class_year,5282.815,912.590
1,off_campus,5279.606,905.648
2,distance_to_campus,5285.333,904.860
3,study,5292.084,903.980
4,gender,5293.265,903.975
5,(null),5289.290,903.764
6,sprinklered,5291.084,903.739
7,gpa,5289.773,903.381
8,greek,5188.615,895.082


Greek is going to be the first predictor

Now to check all 2 parameter models

In [4]:
rows = []

base = smf.glm('claim_count ~ greek', data=pp, family=sm.families.Poisson()).fit()
rows.append(('greek', base.aic,
             poisson_deviance(pp_hold.claim_count, base.predict(pp_hold))))

for v in [x for x in RATING_VARS if x != 'greek']:
    f = f'claim_count ~ greek + {v}'
    m = smf.glm(f, data=pp, family=sm.families.Poisson()).fit()
    rows.append((f'greek + {v}', m.aic,
                 poisson_deviance(pp_hold.claim_count, m.predict(pp_hold))))

step2 = (pd.DataFrame(rows, columns=['model', 'train_AIC', 'holdout_deviance'])
           .sort_values('holdout_deviance', ascending=False)
           .reset_index(drop=True))

step2.round(3)

,model,train_AIC,holdout_deviance
0,greek + class_year,5183.433,903.297
1,greek + off_campus,5177.649,897.045
2,greek + distance_to_campus,5183.982,896.588
3,greek + study,5190.051,895.812
4,greek + gender,5192.647,895.303
5,greek,5188.615,895.082
6,greek + gpa,5189.088,895.047
7,greek + sprinklered,5190.363,894.928


moving foward with greek and sprinklered

In [5]:
rows = []

base = smf.glm('claim_count ~ greek + sprinklered', data=pp,
               family=sm.families.Poisson()).fit()
rows.append(('greek + sprinklered', base.aic,
             poisson_deviance(pp_hold.claim_count, base.predict(pp_hold))))

for v in [x for x in RATING_VARS if x not in ('greek', 'sprinklered')]:
    f = f'claim_count ~ greek + sprinklered + {v}'
    m = smf.glm(f, data=pp, family=sm.families.Poisson()).fit()
    rows.append((f'greek + sprinklered + {v}', m.aic,
                 poisson_deviance(pp_hold.claim_count, m.predict(pp_hold))))

step3 = (pd.DataFrame(rows, columns=['model', 'train_AIC', 'holdout_deviance'])
           .sort_values('holdout_deviance', ascending=False)
           .reset_index(drop=True))

step3.round(3)

,model,train_AIC,holdout_deviance
0,greek + sprinklered + class_year,5185.212,903.141
1,greek + sprinklered + off_campus,5179.408,896.845
2,greek + sprinklered + distance_to_campus,5185.772,896.408
3,greek + sprinklered + study,5191.800,895.664
4,greek + sprinklered + gender,5194.384,895.134
5,greek + sprinklered,5190.363,894.928
6,greek + sprinklered + gpa,5190.861,894.885


Analyze all 4 predictor models

In [6]:
rows = []

base = smf.glm('claim_count ~ greek + sprinklered + gpa', data=pp,
               family=sm.families.Poisson()).fit()
rows.append(('greek + sprinklered + gpa', base.aic,
             poisson_deviance(pp_hold.claim_count, base.predict(pp_hold))))

for v in [x for x in RATING_VARS if x not in ('greek', 'sprinklered', 'gpa')]:
    f = f'claim_count ~ greek + sprinklered + gpa + {v}'
    m = smf.glm(f, data=pp, family=sm.families.Poisson()).fit()
    rows.append((f'greek + sprinklered + gpa + {v}', m.aic,
                 poisson_deviance(pp_hold.claim_count, m.predict(pp_hold))))

step4 = (pd.DataFrame(rows, columns=['model', 'train_AIC', 'holdout_deviance'])
           .sort_values('holdout_deviance', ascending=False)
           .reset_index(drop=True))

step4.round(3)

,model,train_AIC,holdout_deviance
0,greek + sprinklered + gpa + class_year,5185.639,903.190
1,greek + sprinklered + gpa + off_campus,5179.912,896.806
2,greek + sprinklered + gpa + distance_to_campus,5186.298,896.332
3,greek + sprinklered + gpa + study,5192.313,895.678
4,greek + sprinklered + gpa + gender,5194.786,895.114
5,greek + sprinklered + gpa,5190.861,894.885


**Since the 3 predictor model has lower test deviance, we will use these 3 predictors to analyze personal property

Now for Additional Living Expenses

In [7]:
ale = fq_train['Additional Living Expense'][RATING_VARS + ['claim_count']].copy()

for c in ['class_year', 'study', 'greek', 'gender', 'off_campus']:
    ale[c] = ale[c].astype('category')

ale_hold = freq[(freq.coverage == 'Additional Living Expense') &
                (freq.training_holdout == 'holdout')].copy()

for c in ['class_year', 'study', 'greek', 'gender', 'off_campus']:
    ale_hold[c] = ale_hold[c].astype('category')

Defining function to perform foward stepwise selection

In [19]:
def forward_stepwise(train, hold, candidates):
    """Forward selection on out-of-sample Poisson deviance.
    Continues while any addition improves holdout deviance.
    Returns (selected variables, one combined table of every step)."""
    keep     = []
    m_cur    = smf.glm('claim_count ~ 1', data=train,
                       family=sm.families.Poisson()).fit()
    d_cur    = poisson_deviance(hold.claim_count, m_cur.predict(hold))
    all_rows = []
    step     = 0

    while True:
        remaining = [v for v in candidates if v not in keep]
        if not remaining:
            break
        step += 1

        rows = [{'step': step,
                 'model': ' + '.join(keep) or '(null)',
                 'train_AIC': m_cur.aic, 'holdout_deviance': d_cur}]

        results = {}
        for v in remaining:
            f = 'claim_count ~ ' + ' + '.join(keep + [v])
            m = smf.glm(f, data=train, family=sm.families.Poisson()).fit()
            d = poisson_deviance(hold.claim_count, m.predict(hold))
            results[v] = (m, d)
            rows.append({'step': step,
                         'model': ' + '.join(keep + [v]),
                         'train_AIC': m.aic, 'holdout_deviance': d})

        all_rows.extend(rows)

        best = min(results, key=lambda v: results[v][1])

        if results[best][1] >= d_cur:
            break

        keep.append(best)
        m_cur, d_cur = results[best]

    return keep, (pd.DataFrame(all_rows)
                    .sort_values('holdout_deviance')
                    .reset_index(drop=True))

Defined a function to do the foward stepwise selection automatically

In [20]:
ale_keep, ale_path = forward_stepwise(ale, ale_hold, RATING_VARS)
ale_path.round(3)

,step,model,train_AIC,holdout_deviance
0,3,greek + sprinklered,3327.548,594.177
1,2,greek + sprinklered,3327.548,594.177
2,3,greek + sprinklered + gpa,3327.832,594.327
3,3,greek + sprinklered + distance_to_campus,3327.026,595.403
4,1,greek,3326.521,595.637
5,2,greek,3326.521,595.637
6,3,greek + sprinklered + off_campus,3323.595,595.689
7,2,greek + gpa,3326.860,595.789
8,2,greek + distance_to_campus,3326.055,596.843
9,3,greek + sprinklered + gender,3331.946,596.855


Based on above, we will use predictors greek and sprinklered for additional living expense

Now calling the function for guest medical coverage.

In [21]:
gm = fq_train['Guest Medical'][RATING_VARS + ['claim_count']].copy()

for c in ['class_year', 'study', 'greek', 'gender', 'off_campus']:
    gm[c] = gm[c].astype('category')

gm_hold = freq[(freq.coverage == 'Guest Medical') &
               (freq.training_holdout == 'holdout')].copy()

for c in ['class_year', 'study', 'greek', 'gender', 'off_campus']:
    gm_hold[c] = gm_hold[c].astype('category')

In [22]:
gm_keep, gm_path = forward_stepwise(gm, gm_hold, RATING_VARS)

gm_path.round(3)

,step,model,train_AIC,holdout_deviance
0,4,greek + gpa + study,1782.991,343.117
1,3,greek + gpa + study,1782.991,343.117
2,4,greek + gpa + study + class_year,1788.661,343.218
3,2,greek + gpa,1777.189,343.264
4,3,greek + gpa,1777.189,343.264
5,3,greek + gpa + class_year,1782.861,343.344
6,4,greek + gpa + study + sprinklered,1784.929,343.353
7,3,greek + gpa + sprinklered,1779.120,343.516
8,4,greek + gpa + study + gender,1787.909,343.859
9,3,greek + gpa + gender,1782.098,344.033


Based on above, we will use predictors greek, gpa, and study for guest medical

Now to analyze Liability

In [23]:
lia = fq_train['Liability'][RATING_VARS + ['claim_count']].copy()

for c in ['class_year', 'study', 'greek', 'gender', 'off_campus']:
    lia[c] = lia[c].astype('category')

lia_hold = freq[(freq.coverage == 'Liability') &
                (freq.training_holdout == 'holdout')].copy()

for c in ['class_year', 'study', 'greek', 'gender', 'off_campus']:
    lia_hold[c] = lia_hold[c].astype('category')

In [27]:
lia_keep, lia_path = forward_stepwise(lia, lia_hold, RATING_VARS)

lia_path.round(3)

,step,model,train_AIC,holdout_deviance
0,5,greek + distance_to_campus + gpa + sprinklered + off_campus,761.146,156.705
1,6,greek + distance_to_campus + gpa + sprinklered + off_campus,761.146,156.705
2,6,greek + distance_to_campus + gpa + sprinklered + off_campus + study,765.990,157.022
3,5,greek + distance_to_campus + gpa + sprinklered,760.513,157.030
4,4,greek + distance_to_campus + gpa + sprinklered,760.513,157.030
5,4,greek + distance_to_campus + gpa + off_campus,759.559,157.033
6,5,greek + distance_to_campus + gpa + sprinklered + study,765.359,157.356
7,3,greek + distance_to_campus + gpa,758.950,157.379
8,4,greek + distance_to_campus + gpa,758.950,157.379
9,4,greek + distance_to_campus + gpa + study,763.777,157.669


IMPORTANT NOTE: the above results suggest that we should use parameters (Greek, distance to campus, gpa, sprinklered, and off-campus). However this now made obvious something we did not initially consider which is there is likely high multicollinearity between distance to campus and off-campus. 

Analyze the VIF to check for multicollinearity (Note: consider running PCA on Liability later and compare models)

In [28]:
off = (pp.off_campus == 'Off campus').astype(float)
r   = np.corrcoef(off, pp.distance_to_campus)[0, 1]

print(f'r = {r:.4f}')
print(f'VIF = {1 / (1 - r**2):.2f}')

r = 0.6420
VIF = 1.70


This VIF does not warrant removing either predictor, and thhough there is likely multicollinearity between them, evidence suggest including them both will improve model predictions 

*So we will move foward using Greek, distance to campus, gpa, sprinklered, and off-campus as out predictors for liabilty

CONCLUSIONS: coefficent table

In [30]:
import re

def clean_term(t):
    m = re.match(r'^(.*?)\[T\.(.*)\]$', t)
    return f"{m.group(1)} = {m.group(2)}" if m else t

FREQ_SPECS = {
    'Personal Property':         ['greek', 'sprinklered', 'gpa'],
    'Additional Living Expense': ['greek', 'sprinklered'],
    'Guest Medical':             ['greek', 'gpa', 'study'],
    'Liability':                 ['greek', 'distance_to_campus', 'gpa',
                                  'sprinklered', 'off_campus'],
}

CAT_VARS = ['class_year', 'study', 'greek', 'gender', 'off_campus']
CATS     = {c: pd.Categorical(freq[c]).categories for c in CAT_VARS}

rows, FREQ_FINAL = [], {}

for c, v in FREQ_SPECS.items():
    d = fq_train[c][RATING_VARS + ['claim_count']].copy()
    for k in CAT_VARS:
        d[k] = pd.Categorical(d[k], categories=CATS[k])

    m = smf.glm('claim_count ~ ' + (' + '.join(v) if v else '1'),
                data=d, family=sm.families.Poisson()).fit()
    FREQ_FINAL[c] = m

    for t in m.params.index:
        rows.append({'coverage': c, 'term': clean_term(t),
                     'coef': m.params[t], 'pval': m.pvalues[t], 'se': m.bse[t],
                     'component': 'frequency'})

freq_coefs = pd.DataFrame(rows)[['coverage', 'term', 'coef', 'pval', 'se', 'component']]
freq_coefs.to_csv(PROC / "freq_coefficients.csv", index=False)

freq_coefs.round(4)

,coverage,term,coef,pval,se,component
0,Personal Property,Intercept,-1.6427,0.0000,0.1275,frequency
1,Personal Property,greek = Non-greek,-0.7964,0.0000,0.0748,frequency
2,Personal Property,sprinklered = True,0.0374,0.6346,0.0787,frequency
3,Personal Property,gpa,-0.0515,0.2191,0.0419,frequency
4,Additional Living Expense,Intercept,-2.3046,0.0000,0.1089,frequency
5,Additional Living Expense,greek = Non-greek,-0.7520,0.0000,0.1022,frequency
6,Additional Living Expense,sprinklered = True,-0.1031,0.3207,0.1039,frequency
7,Guest Medical,Intercept,-3.1844,0.0000,0.2707,frequency
8,Guest Medical,greek = Non-greek,-0.5928,0.0002,0.1590,frequency
9,Guest Medical,study = Humanities,0.0302,0.8819,0.2032,frequency
